# DLM on Colab

Thin runner: this notebook only launches the scripts in the repo (`run.py`). All logic lives in `dlm/`, so the laptop and Colab run identical code.

Run the cells top to bottom. Phases already finished on Drive are skipped (each phase is idempotent). For pre-training choose **Runtime -> Change runtime type -> GPU (L4 or A100: bf16)** first.

**The tokenizer comes from the repo (`tokenizer/`, committed), not from retraining here**, so Colab and the laptop use exactly the same one. The dataset revision is pinned for the same reason.


In [ ]:
# 1. Google Drive (data and checkpoints persist here across Colab sessions)
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/diffusion_lm_outputs"


In [ ]:
# 2. Get the code from GitHub (public repo; tokenizer/ is committed there)
REPO = "https://github.com/bitlabsdevteam/Diffusion-Language-Model.git"
import os
if os.path.isdir("/content/dlm_repo"):
    !git -C /content/dlm_repo pull --ff-only
else:
    !git clone $REPO /content/dlm_repo
%cd /content/dlm_repo
!ls tokenizer


In [ ]:
# 3. Dependencies: install ONLY what is missing, never numpy/torch/transformers (reinstalling them can break Colab's live kernel).
# Training itself needs only torch, numpy and matplotlib, which Colab ships. Phases 0 and 2 (below) also need these four:
import importlib.util, subprocess, sys
missing = [p for p in ("datasets", "huggingface_hub", "pyarrow", "tokenizers") if importlib.util.find_spec(p) is None]
print("missing:", missing or "none")
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)


### Data: two ways (choose one)

**A. Build it on Colab** (cells 4-6): downloads TinyStories at the pinned revision and tokenizes it (about 2 GB download; the tokenizing took about 4 minutes on a laptop CPU, probably longer on Colab's CPU).

**B. Upload the finished cache and skip cells 4-6**: copy `DLM/data/tokens/` (970 MB) from your computer to Drive as `MyDrive/diffusion_lm_outputs/data/tokens/`. Training reads only that cache (cell 7 copies it to the local disk), so Phases 0 and 2 never need to run on Colab. It was built with the same committed tokenizer, so the results are identical.


In [ ]:
# 4. Phase 0: download TinyStories at the pinned revision (writes to Drive: diffusion_lm_outputs/data; a second run does nothing)
REV = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"
!python run.py phase0 --revision $REV


In [ ]:
# 5. Check the repo's tokenizer loads here (it is NOT retrained on Colab)
from tokenizers import Tokenizer
tok = Tokenizer.from_file("tokenizer/tokenizer.json")
print("vocab size:", tok.get_vocab_size(), "| tokens:", tok.encode("Once upon a time", add_special_tokens=False).tokens)


In [ ]:
# 6. Phase 2: tokenize + block the Stories with the repo's tokenizer (writes data/tokens on Drive; a second run does nothing)
!python run.py phase2


In [ ]:
# 7. Copy the token cache from Drive to the local Colab disk: training reads random blocks all the time and Drive is slow
!mkdir -p /content/tokens && cp -r $OUT/data/tokens/. /content/tokens/ && du -sh /content/tokens


## Phase 5: pre-training

Checkpoints and `final/` go to Drive (`diffusion_lm_outputs/checkpoints/...`), so a disconnected session continues where it stopped: the cells below always pass `--resume` (it starts from scratch when there is no checkpoint, and refuses to resume if the recipe changed). Watch the first evaluation lines before leaving it running: the loss may sit on the ~5.92 token-frequency plateau for a while. The starting values come from the old notebooks' proxy trials and have never been run to completion on this model. bf16 is used on an L4/A100; a T4 falls back to slow fp32.


In [ ]:
# 8. Live monitoring (run this BEFORE the training cells; it stays on screen while training runs in the next cell):
# TensorBoard shows cross-entropy loss, learning rate, epoch, gradient norm and the validation / train-eval curves.
%load_ext tensorboard
%tensorboard --logdir $OUT/checkpoints


In [ ]:
# 9. Quick GPU check: a 200-step run of the tiny model (about a minute)
!python run.py pretrain --size tiny --steps 200 --eval-every 100 --warmup 20 --cache /content/tokens --out $OUT/checkpoints/smoke --resume


In [ ]:
# 9b. Sample from the smoke run (first run of the sampler on CUDA; the tiny model is barely trained, so expect soup)
!python run.py generate --ckpt $OUT/checkpoints/smoke/final --n 2 --max-new-tokens 96 --seed 0


In [ ]:
# 10. The real run (40,000 steps). Safe to re-run after a disconnect: --resume continues from the last resume.pt.
!python run.py pretrain --cache /content/tokens --out $OUT/checkpoints/pretrain_base --resume


In [ ]:
# 10b. Generate Stories from the trained model: unconditional, then continuing your opening words.
# --trace shows the unmasking as text rows, --gif writes an animation (shown in the next cell).
!python run.py generate --ckpt $OUT/checkpoints/pretrain_base/final --n 3 --seed 0
!python run.py generate --ckpt $OUT/checkpoints/pretrain_base/final --prompt "Once upon a time, there was a little" --n 2 --seed 0 --trace --gif /content/unmask.gif


In [ ]:
# 10c. Show the unmasking animation
from IPython.display import Image, display
display(Image("/content/unmask_1.gif"))


In [ ]:
# 10d. Evaluate the trained model: validation likelihood (nats/token, bits/char), sample statistics against real Stories,
# GPT-2 reference perplexity (downloads ~500 MB once). Writes eval/report.md and eval/samples.md next to the run.
!python run.py eval --ckpt $OUT/checkpoints/pretrain_base/final --cache /content/tokens


In [ ]:
# 10e. The decoding sweep: 24 settings (tokens per step x blocks/global x linear/cosine), 50 Stories each. LONG: run it only
# after 10d looks sensible. Add --sweep-n 20 for a faster pass.
!python run.py eval --ckpt $OUT/checkpoints/pretrain_base/final --cache /content/tokens --sweep --out $OUT/checkpoints/pretrain_base/eval_sweep


In [ ]:
# 10f. Read the report and the samples (the samples are what tell you whether the Stories are any good)
print(open(f"{OUT}/checkpoints/pretrain_base/eval/report.md").read())
print(open(f"{OUT}/checkpoints/pretrain_base/eval/samples.md").read()[:6000])


In [ ]:
# 11. Latest training curves (cross-entropy loss, learning rate, epoch, grad norm); also refreshed during the run in
# diffusion_lm_outputs/checkpoints/pretrain_base/training_curves.png, or redraw with: !python run.py plot --out $OUT/checkpoints/pretrain_base
from IPython.display import Image, display
display(Image(f"{OUT}/checkpoints/pretrain_base/training_curves.png"))


In [ ]:
# 12. Offline tests (optional)
!python run.py test


## Later phases (not built yet)

`python run.py sft` (Phase 8) will be added as a cell here once it exists; see `PRD.md`.
